# Lab Exercise 1. Scraping Static Websites


This is the warmup task for the first laboratory exercise. It consists of scraping static Websites with BeautifulSoap.

 It should be completed at home and presented at the laboratory.

**Total points: 2**

### Task Description

Scrape the information about the products on the following page:
https://clevershop.mk/product-category/mobilni-laptopi-i-tableti/

For each product scrape:


*   Product title (selector `'.wd-entities-title'`)
*   Product regular price (selector `'.woocommerce-Price-amount'`)
*   Product discount price (if available), same selector as regular price
*   URL to the product page
*   Add to cart button URL

***Help: There are multiple product pages, for each page you need to send a separate request***


Save the results as a DataFrame object

You can add as many code cells as you need.

________________________________________________________________

### Requirements

Import libraries and modules that you are going to use

In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Send HTTP request to the target Website

In [ ]:
url= "https://clevershop.mk/product-category/mobilni-laptopi-i-tableti/"
response = requests.get(url)

check the response status code

In [ ]:
response.status_code

200

### Parse the response content with BeautifulSoap

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

### Extract data from the BeautifulSoap object using any selectors, attribute identifiers, etc.

* Product title (selector '.wd-entities-title')
* Product regular price (selector '.woocommerce-Price-amount')
* Product discount price (if available), same selector as regular price
* URL to the product page
* Add to cart button URL

# *If i were to extract data from only the first page, I would have done the following:*

In [ ]:
products = soup.select('.product-wrapper')
parsed_products = []
for product in products:
  name = product.select_one('.wd-entities-title').text.strip()

  regular_price = product.select_one('.woocommerce-Price-amount').text.strip()

  discount = product.select_one('.onsale')
  if discount is not None:
    discount = discount.text.strip()

  link = product.select_one('.wd-entities-title a').get("href")

  add_to_cart_link = product.select_one('.wd-add-btn a').get("href")

  product_dict = {
      "ProductName": name,
      "RegularPrice": regular_price,
      "DiscountPercent": discount,
      "ProductPageLink": link,
      "CartButtonLink": add_to_cart_link
  }

  parsed_products.append(product_dict)

Repeat the extraction process for each page of products

# *Now, that I have to extract data from all of the pages, I made changes to the previous code, accordingly:*


I took the previuos code and made a function, that will be called as many times as there are pages on the WebPage

In [ ]:
url= "https://clevershop.mk/product-category/mobilni-laptopi-i-tableti/page/"

In [ ]:
def extract_product_as_dict(product):
  name = product.select_one('.wd-entities-title').text.strip()

  regular_price = product.select_one('.woocommerce-Price-amount').text.strip()

  discount = product.select_one('.onsale')
  if discount is not None:
    discount = discount.text.strip()

  link = product.select_one('.wd-entities-title a').get("href")

  add_to_cart_link = product.select_one('.wd-add-btn a').get("href")

  product_dict = {
      "ProductName": name,
      "RegularPrice": regular_price,
      "DiscountPercent": discount,
      "ProductPageLink": link,
      "CartButtonLink": add_to_cart_link
  }

  return product_dict

In [ ]:
parsed_products = []
for i in range (1,15):

  new_url = url + str(i)
  response = requests.get(new_url)
  soup = BeautifulSoup(response.text, "html.parser")
  products = soup.select('.product-wrapper')
  for product in products:
    result = extract_product_as_dict(product)
    parsed_products.append(result)

In [ ]:
len(parsed_products)

320

### Create a pandas DataFrame with the scraped products

In [ ]:
df = pd.DataFrame(parsed_products)

In [ ]:
df

,ProductName,RegularPrice,DiscountPercent,ProductPageLink,CartButtonLink
0,Acer A315-23-A7KD,17.590 ден,None,https://clevershop.mk/product/acer-a315-23-a7kd/,?add-to-cart=21494
1,Acer A315-23-R5P2,27.490 ден,None,https://clevershop.mk/product/acer-a315-23-r5p2/,?add-to-cart=21510
2,ACER Aspire 1 A115-22,18.999 ден,-16%,https://clevershop.mk/product/acer-aspire-1-nx...,?add-to-cart=20826
3,Acer Aspire 3 A315-23-R26A,29.990 ден,None,https://clevershop.mk/product/acer-aspire-3-a3...,?add-to-cart=21516
4,Acer Aspire 3 A315-58-33WK,24.490 ден,None,https://clevershop.mk/product/21498/,?add-to-cart=21498
...,...,...,...,...,...
315,Monitor 27 Philips 272E1GAJ/00 VA 1ms 144Hz,12.890 ден,None,https://clevershop.mk/product/monitor-27-phili...,?add-to-cart=12618
316,Philips 24″ 243V7QDSB,8.390 ден,None,https://clevershop.mk/product/philips-24%e2%80...,?add-to-cart=12396
317,Philips 27″ 278E1A/00 4K UHD IPS,18.990 ден,None,https://clevershop.mk/product/hp-27%e2%80%b3-2...,?add-to-cart=12218
318,Philips 279C9-00 MON LED 27″ 3840 x 2160 5Ms 6...,26.990 ден,None,https://clevershop.mk/product/philips-279c9-00...,?add-to-cart=12578


Save the dataframe as `.csv`

In [ ]:
df.to_csv('products-clevershop.csv', index=False)